# 🎯 Objetivos

El objetivo de este script es construir y evalaur el modelo de recomendación colaborativo basado en usuarios donde se recomeiendan videojuegos a partir del comportameinto de otros jugadores con comportamientos similares.

* ⚙️ **Configuración inicial**. Rutas, paquetes y funciones.
* ♻️ **Carga de datos**.
* ✅ **Chequear la integridad**.
* 🧊 **Construir la matriz usuario-juego**.
* 📈 **Evaluación del modelo**.
* 💾 **Guardado de información**.

# ⚙️ Configuración

## Rutas (paths)

In [1]:
import os

# Rutas en función de la carpeta raíz del proyecto (README.md)
base_path = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in locals() else os.path.abspath("..")

# Subrutas
data_dir = os.path.join(base_path, "data")
db_path = os.path.join(data_dir, "steam.db")
src_dir = os.path.join(base_path, "src")
models_dir = os.path.join(base_path, "models")
models_collab_dir = os.path.join(models_dir, "collaborative")
results_dir = os.path.join(base_path, "res")

## Paquetes y funciones

In [ ]:
# Sistema y configuración
import warnings
warnings.filterwarnings("ignore")
import sqlite3

# Modelos
import joblib

# Análisis y manipulación de datos
import pandas as pd
import numpy as np

# Matrices y transformaciones
from scipy import sparse
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

# Modelos de machine learning
from sklearn.neighbors import NearestNeighbors

# Funciones personalizadas    
from utils.evaluation_functions import evaluate_collaborative_LMO_at_k

# ♻️ Carga de datos

In [3]:
# Tabla interacciones
conn = sqlite3.connect(db_path)
cur = conn.cursor()

user_games = pd.read_sql("SELECT * FROM user_games_final", sqlite3.connect(db_path))

conn.close()

# Hacemos una copia para trabajar
user_games = user_games.copy()

In [4]:
# Tablas test - train
conn = sqlite3.connect(db_path)
cur = conn.cursor()

user_train = pd.read_sql("SELECT * FROM user_train_LMO", sqlite3.connect(db_path))
user_test = pd.read_sql("SELECT * FROM user_test_LMO", sqlite3.connect(db_path))

conn.close()

# Hacemos una copia para trabajar
user_train = user_train.copy()
user_test = user_test.copy()

In [5]:
# Normalización de tipos
user_games["user_id"] = user_games["user_id"].astype(np.int64)
user_games["appid"]   = user_games["appid"].astype(np.int64)

user_train["user_id"] = user_train["user_id"].astype(np.int64)
user_train["appid"]   = user_train["appid"].astype(np.int64)

user_test["user_id"]  = user_test["user_id"].astype(np.int64)
user_test["appid"]    = user_test["appid"].astype(np.int64)

In [6]:
# Índice de juegos
idx = pd.read_csv(os.path.join(models_dir, "games_index.csv")) 

In [7]:
uids = user_train["user_id"].unique()
gids = user_train["appid"].unique()

user_map = {u: i for i, u in enumerate(uids)}
game_map = {g: j for j, g in enumerate(gids)}
inv_game_map = np.array(gids)

# Alinear test con juegos presentes en train
valid_appids_cf = set(gids)
user_test = user_test[user_test["appid"].isin(valid_appids_cf)].copy()


# ✅ Chequeos

En este apartado se estudia la estructura de la tabla user_game y la división en train y test con el objetivo de verificar su calidad para crear el sistema de recomendación colaborativo.

In [8]:
# Columnas y tipos
print(user_games.dtypes)
print(user_train.dtypes)
print(user_test.dtypes)

user_id    int64
appid      int64
dtype: object
user_id    int64
appid      int64
dtype: object
user_id    int64
appid      int64
dtype: object


In [9]:
print(user_train.groupby("user_id")["appid"].count().describe())
print(user_train.groupby("appid")["user_id"].count().describe())

count    47627.000000
mean       143.777059
std        169.566338
min          5.000000
25%         42.000000
50%         89.000000
75%        178.000000
max       2289.000000
Name: appid, dtype: float64
count     2468.000000
mean      2774.582658
std       3627.093467
min         53.000000
25%        562.750000
50%       1507.000000
75%       3461.500000
max      32785.000000
Name: user_id, dtype: float64


# 🧊 Construcción de la matriz usuario-juego

El objetivo de este apartado es crear la matriz de las interacciones entre usuarios y videojuegos para el modelo colaborativo. 

In [10]:
# Mapear usuarios (u) y juegos (g) a índices numéricos
uids = user_train["user_id"].unique()
gids = user_train["appid"].unique()

user_map = {u: i for i, u in enumerate(uids)}
game_map = {g: j for j, g in enumerate(gids)}

# Mapeo indice - appid
inv_game_map = np.array(gids)


In [11]:
# Construir la matriz dispersa (CSR)
rows = user_train["user_id"].map(user_map).to_numpy(dtype=np.int32)
cols = user_train["appid"].map(game_map).to_numpy(dtype=np.int32)
data = np.ones(len(user_train), dtype=np.float32)

R = csr_matrix((data, (rows, cols)), shape=(len(uids), len(gids)))
print(f"Matriz R: {R.shape[0]} usuarios × {R.shape[1]} juegos  |  densidad {R.nnz / (R.shape[0]*R.shape[1]):.6f}")

Matriz R: 47627 usuarios × 2468 juegos  |  densidad 0.058257


In [12]:
# Penalizar juegos muy populares
item_freq = np.array((R > 0).sum(axis=0)).ravel()
idf = np.log1p(R.shape[0] / (item_freq + 1))  # ponderación inversa de frecuencia
idf_diag = csr_matrix(np.diag(idf))

# Aplicar la ponderación (R ponderada por IDF)
R_idf = R @ idf_diag

In [13]:
# Normalizar filas para similitud coseno
R_norm = normalize(R_idf, norm='l2', axis=1, copy=True)

In [14]:
n_neighbors = 70  # número de vecinos más similares por usuario
knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=n_neighbors + 1)
knn.fit(R_norm)

,n_neighbors,71
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


# 📈 Evaluación del modelo

In [15]:
evaluate_collaborative_LMO_at_k(
    k=10,
    user_train=user_train,
    user_test=user_test,
    knn=knn,
    R_norm=R_norm,    
    R=R,
    user_map=user_map,
    inv_game_map=inv_game_map,
    idx=idx,
    results_dir=results_dir,
    tag="20251130"
)

{'k': 10,
 'precision@k': 0.07285993239129067,
 'recall@k': 0.24286644130430218,
 'f1@k': 0.11209220367890868,
 'ndcg@k': 0.35222178470896565,
 'hit_rate@k': 0.5424864047704033,
 'item_coverage@k': 0.6073743922204214,
 'num_users': 47627}

In [16]:
evaluate_collaborative_LMO_at_k(
    k=50,
    user_train=user_train,
    user_test=user_test,
    knn=knn,
    R_norm=R_norm,
    R=R,
    user_map=user_map,
    inv_game_map=inv_game_map,
    idx=idx,
    results_dir=results_dir,
    tag="20251130"
)

{'k': 50,
 'precision@k': 0.028330148865139525,
 'recall@k': 0.4721691477523254,
 'f1@k': 0.053453111066300976,
 'ndcg@k': 0.39451829330868154,
 'hit_rate@k': 0.8256451172654167,
 'item_coverage@k': 0.8403565640194489,
 'num_users': 47627}

# 💾 Guardar resultados

In [17]:
# Modelo KNN entrenado
knn_model_path = os.path.join(models_collab_dir, f"knn_user_based_{n_neighbors:.1f}.pkl")
joblib.dump({
    "knn": knn,
    "user_map": user_map,
    "game_map": game_map,
    "inv_game_map": inv_game_map
}, knn_model_path)

# Guardar matrices
R_path = os.path.join(models_collab_dir, "R_user_based.npz")
R_norm_path = os.path.join(models_collab_dir, "R_norm_user_based.npz")

sparse.save_npz(R_path, R)
sparse.save_npz(R_norm_path, R_norm)